# UniProt RAG — query & summarize

Pipeline: **ingest** → **index** → **search** → **summarize** (search and summarize run in this notebook).

Prerequisites: `python src/ingest_uniprot.py` then `python src/build_uniprot_index.py`. EDA: `uniprot_eda.ipynb`.

In [23]:
# Natural-language query
query = "mismatch repair mechanism"

In [22]:
# Optional: install dependencies if needed
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [25]:
# Imports
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer


In [26]:
# Project paths and shared RAG helpers
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from rag_search import search
from rag_summary import load_llm, summarize
from rag_export import append_rag_report


## Search

**Ranking:** Chroma returns `TOP_K_CHUNKS` chunks. Each protein keeps its closest chunk as its score; the top `TOP_K_RESULTS` proteins by that score are selected. The LLM receives one full parquet row per protein (no chunk text).

In [24]:
# Retrieval limits
TOP_K_CHUNKS = 20  # chunk pool from Chroma; used to rank proteins (not sent to LLM)
TOP_K_RESULTS = 3  # top proteins by best-chunk distance; LLM gets 1 full parquet row each

In [ ]:
# Paths, embedding model, search config
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
UNIPROT_PARQUET = data_processed / "uniprot_rag.parquet"
UNIPROT_COLLECTION = "protein_chunks"
UNIPROT_RECORD_FIELDS = [
    "Entry", "Entry Name", "Gene Names", "Protein names", "Length",
    "Function [CC]", "Involvement in disease", "Subcellular location [CC]",
    "Domain [FT]", "Region", "Zinc finger", "Domain [CC]",
    "Natural variant", "Polymorphism", "Tissue specificity",
    "Developmental stage", "Interacts with",
]

# Maps Chroma chunk metadata and parquet columns to rag_search.search()
SEARCH_CONFIG = {
    "collection_name": UNIPROT_COLLECTION,  # Chroma collection to query
    "parquet_path": UNIPROT_PARQUET,        # full records loaded for top-ranked proteins
    "group_key": "entry_name",              # metadata field: collapse chunks to one protein
    "record_id_column": "Entry",            # parquet column used to look up full records
    "record_id_meta_key": "accession",      # chunk metadata field holding the same ID
    "record_fields": UNIPROT_RECORD_FIELDS, # parquet columns included in LLM context
    "section_header_template": "### Result {rank} — entry_name={entry_name}, accession={accession}, gene={gene}",  # per-result header; placeholders from metadata
    "hit_fields": [("entry_name", "entry_name"), ("accession", "accession"), ("gene", "gene")],  # metadata keys printed for each hit
    "full_record_label": "Full UniProt record",       # label above the joined parquet row
    "selected_label": "protein record(s) for context",  # verbose log when loading records
}


In [35]:
embed_model = SentenceTransformer(EMBED_MODEL)
# returns ranked hits + context string for the LLM
result = search(
    query,
    SEARCH_CONFIG,
    top_k_chunks=TOP_K_CHUNKS,
    top_k_results=TOP_K_RESULTS,
    chroma_dir=chroma_dir,
    embed_model=embed_model,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection protein_chunks: 19201 chunks

RETRIEVAL
Query: mismatch repair mechanism
Chroma returned the top 20 matching chunks (20 chunks across 8 unique record(s)).
Top 3 record(s) for the LLM: among proteins in this pool, rank by each protein's best (lowest-distance) chunk, then take the top 3. Chunks are used for ranking only — the LLM gets one full parquet row per selected record.
(13 chunk(s) in the pool belong to proteins outside the top 3.)

TOP 3 RECORD(S) (ranked by best-matching chunk per record)
Preview below is the closest chunk only (truncated to 500 chars).

1. entry_name=MSH6_HUMAN accession=P52701 gene=MSH6 best_chunk=function dist=0.5097 (2 retrieved chunk(s))
Entry name: MSH6_HUMAN
Accession: P52701
Gene: MSH6
Protein: DNA mismatch repair protein Msh6 (hMSH6) (G/T mismatch-binding protein) (GTBP) (GTMBP) (MutS protein homolog 6) (MutS-alpha 160 kDa subunit) (p160)
Length: 1360
Type: function
Component of the post-replicative DNA mismatch repair system (MMR). Heterodim

## Summarize


In [ ]:
SYSTEM_PROMPT = (
    "You are a protein biology assistant. "
    "Write a brief summary in plain English sentences that answers the user's query using ONLY the UniProt records provided. "
    "Always cite the Uniprot Entry Name and Protein Name in your answer. "
    "Do not invent proteins, accessions, variants, or other facts not present in the provided text. "
    "If information is missing, say so briefly."
)

# SYSTEM_PROMPT_1 = (
#     "You are a protein biology assistant. Summarize UniProt protein search results "
#     "for the user's query using ONLY the full records provided. "
#     "Be concise, accurate, and structured. For each relevant protein, mention entry name, "
#     "accession, gene, function, disease involvement, and tissue expression when available. "
#     "If information is missing, say so. Do not invent facts."
# )

Load summarizer → generate answer from retrieved context only.

- **local:** downloads once → cached under `~/.cache/huggingface/`; 0.5B ~2GB RAM.
- **openai:** calls `gpt-4o-mini` via API; set `OPENAI_API_KEY` in `.env`.

The LLM receives the top 3 UniProt records, not the raw chunk pool.

In [ ]:
# Summarization backend: "local" or "openai"
SUMMARIZER_BACKEND = "local"
# Local model (SUMMARIZER_BACKEND == "local")
# Needs ~8GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# Needs ~4GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# OpenAI model (SUMMARIZER_BACKEND == "openai")
API_MODEL = "gpt-4o-mini"
MAX_NEW_TOKENS = 1024
REPORT_PATH = project_root / "reports/rag_log.md"
CSV_LOG_PATH = project_root / "data/reports/rag_log.csv"

In [36]:
if SUMMARIZER_BACKEND == "local":
    llm_model, llm_tokenizer = load_llm(
        LLM_MODEL,
        model=globals().get("llm_model"),
        tokenizer=globals().get("llm_tokenizer"),
    )
else:
    llm_model, llm_tokenizer = None, None


LLM already loaded: Qwen/Qwen2.5-0.5B-Instruct


In [37]:
summary = summarize(
    query,
    result["context"],
    llm_model,
    llm_tokenizer,
    SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    backend=SUMMARIZER_BACKEND,
    api_model=API_MODEL,
)
print("=" * 60)
print(f"Query: {query}\n")
print(summary)

append_rag_report(
    REPORT_PATH,
    query,
    result,
    summary,
    source_name="uniprot",
    summarizer_backend=SUMMARIZER_BACKEND,
    model_name=API_MODEL if SUMMARIZER_BACKEND == "openai" else LLM_MODEL,
    csv_path=CSV_LOG_PATH,
)


Query: mismatch repair mechanism

The MSH3 protein interacts with several genes, including DUC1, DUG, and MSH2. Specifically, it forms a heterodimer with MSH2 to bind to DNA mismatches, initiating DNA repair. The interaction involves the regions 31-122, where there is a disordered region due to the presence of natural variants.


In [1]:
# import re
# re.findall(r"accession=([A-Z0-9]+)", result["context"])
# # ['P52701', 'P43246', 'P20585']

# # retrieval correctly sent MSH6, MSH2, and MSH3; the answer only discussed P52701 because the tiny LLM didn’t use the full context well.